# weight-decay-l2-add composite — cx14: training loop with L2 WD step + set_to_none zero_grad

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `weight-decay-l2-add`, `zero-grad-set-none`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "weight-decay-l2-add"
DD_ATOM_IDS = ["weight-decay-l2-add", "zero-grad-set-none"]
DD_SUBTOPICS = ["Optimizer: Weight decay L2", "PyTorch: zero_grad"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Every training step has the same outer rhythm: `forward -> backward -> step -> zero_grad`. The step folds in **L2 weight decay**, and the zero_grad uses **set_to_none semantics** so the next backward doesn't accidentally accumulate on stale data.

**Atom A — weight-decay-l2-add.** `g <- g + lam * theta`, then `theta <- theta - lr * g`.

**Atom B — zero-grad-set-none.** Setting `p.grad = None` (rather than `p.grad.zero_()`) is the modern PyTorch default. Why: (i) skips the memset-to-zero kernel; (ii) makes a subsequent `+=` into `p.grad` a write (allocates a fresh tensor) instead of an accumulate. After `zero_grad(set_to_none=True)`, `backward()` REASSIGNS `p.grad`, it does not in-place-add.

**Anatomy of one outer step.**
```python
loss = f(x, target).mean()
loss.backward()                               # populates p.grad
for p in params:                              # the 'step'
    g = p.grad + lam * p                      # Atom A
    p.data.add_(g, alpha=-lr)
for p in params: p.grad = None                # Atom B
```

**Why both atoms together.** A common bug: forgetting `zero_grad` keeps adding new grads to old ones, so the WD-augmented step keeps drifting. A second common bug: zeroing with `p.grad.zero_()` while expecting `set_to_none` semantics — your invariant 'p.grad is None at the top of each step' silently fails.

### Composite Exercise — training loop with L2 WD step + set_to_none zero_grad

**Atoms exercised together**: `weight-decay-l2-add`, `zero-grad-set-none`

Implement `cx14_train_loop(x, target, params, lr, weight_decay, n_steps)`:

On each of `n_steps` iterations:
1. Compute `pred = x @ params[0]` (params is a single-element list `[W]`, shape `(d, k)`).
2. `loss = ((pred - target) ** 2).mean()`.
3. `loss.backward()`.
4. **L2 WD step**: for `p in params`, `g = p.grad + weight_decay * p.data`, then `p.data.add_(g, alpha=-lr)` (no momentum here — vanilla SGD + WD).
5. **set_to_none zero_grad**: for `p in params`, set `p.grad = None`.

Return the FINAL loss as a float.

The test checks: (a) FINAL value of `W` matches `torch.optim.SGD(lr=lr, weight_decay=lam, momentum=0)`; (b) AFTER the loop, `params[0].grad is None` (not just zero); (c) the training actually drove the loss down (sanity).

In [ ]:
def cx14_train_loop(x, target, params, lr, weight_decay, n_steps):
    last_loss = float('nan')
    for _ in range(n_steps):
        pred = x @ params[0]
        loss = ((pred - target) ** 2).mean()
        loss.backward()
        # Atom A (weight-decay-l2-add): fold lam*theta into the grad, then update.
        for p in params:
            g = p.grad + weight_decay * p.data
            p.data.add_(g, alpha=-lr)
        # Atom B (zero-grad-set-none): reset by setting to None, not by zeroing in-place.
        for p in params:
            p.grad = None
        last_loss = loss.item()
    return last_loss


<details><summary>Show solution — cx14</summary>

```python
def cx14_train_loop(x, target, params, lr, weight_decay, n_steps):
    last_loss = float('nan')
    for _ in range(n_steps):
        pred = x @ params[0]
        loss = ((pred - target) ** 2).mean()
        loss.backward()
        # Atom A (weight-decay-l2-add): fold lam*theta into the grad, then update.
        for p in params:
            g = p.grad + weight_decay * p.data
            p.data.add_(g, alpha=-lr)
        # Atom B (zero-grad-set-none): reset by setting to None, not by zeroing in-place.
        for p in params:
            p.grad = None
        last_loss = loss.item()
    return last_loss
```

Three subtleties: (1) `g = p.grad + weight_decay * p.data` makes a NEW tensor so the next iteration's backward can safely reassign `p.grad`. (2) `p.grad = None` (not `p.grad.zero_()`) is the set_to_none flavour PyTorch defaults to since 1.7 — backward reassigns the slot. (3) Computing `loss.item()` inside the loop (not just at the end) is fine; the return value is just the most recent. NB: `p.data.add_(g, alpha=-lr)` is the vectorised form of `p.data -= lr * g` and avoids building an intermediate.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx14'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx14',
        'subtopics': ["Optimizer: Weight decay L2", "PyTorch: zero_grad"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()